# BPMN Assistant - Stage 1: SFT (QLoRA) [Kaggle]

Trains the SFT LoRA adapter on `data/instruction/*` (capabilities C1-C4, C6, C7).

## Before running
1. Attach your **bpmn-training-data** dataset (*Add Input*). `DATA_DIR` is auto-detected.
2. Settings -> Accelerator = **GPU T4 x2**, Internet = **On**. Run All.

## After it finishes (IMPORTANT)
**Save Version -> Save & Run All (Commit)** to persist `/kaggle/working/bpmn-sft-adapter` as this
notebook's Output; Stage 2 (DPO) attaches it as input. Single-GPU + `MAX_LEN=1536` keep 8B QLoRA in 16 GB.

In [1]:
# 1) Dependencies
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use ONE T4 - stops HF Trainer DataParallel across both GPUs (OOM cause)
!pip install -q -U "transformers>=4.51" "trl==0.21.0" "peft>=0.13" "datasets>=2.20" "bitsandbytes>=0.44" "accelerate>=1.0"
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    print("No HF_TOKEN secret - continuing unauthenticated (fine for public Qwen).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 10.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 80.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 42.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 21.1 MB/s eta 0:00:00
torch 2.10.0+cu128 | transformers 5.15.0 | trl 0.21.0
CUDA: True Tesla T4
No HF_TOKEN secret - continuing unauthenticated (fine for public Qwen).


In [2]:
# 2) Config
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"          # If you keep OOMing on the T4, use "Qwen/Qwen2.5-3B-Instruct".
OUTPUT_DIR = "/kaggle/working"
MAX_LEN    = 1024                     # single-T4 safe for 8B QLoRA. Lower to 1024 if OOM; raise to 2048 if headroom.
ENABLE_THINKING = False
SFT_EPOCHS = 2
SYSTEM_PROMPT = 'You are a BPMN 2.0 expert assistant. Answer precisely and follow BPMN 2.0 conventions. When asked to generate a diagram, output valid BPMN 2.0 XML. When asked to review a diagram, identify concrete issues and how to fix them.'
import os, glob
_hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert _hits, "sft_train.jsonl not found under /kaggle/input. Present: " + str(os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")
DATA_DIR = os.path.dirname(_hits[0])
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /kaggle/input/datasets/lalitamittal/training


In [3]:
# 3) Tokenizer + SFT data (thinking OFF) + length filter (drop, do not truncate long XML)
import re, numpy as np
from transformers import AutoTokenizer
from datasets import load_dataset
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
sft = load_dataset("json", data_files={"train": f"{DATA_DIR}/sft_train.jsonl", "val": f"{DATA_DIR}/sft_val.jsonl"})
def render(ex):
    t = tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=ENABLE_THINKING)
    t = re.sub(r"<think>\s*</think>\s*", "", t)   # strip empty think blocks Qwen3 may inject
    return {"text": t}
sft = sft.map(render, remove_columns=sft["train"].column_names)
def _ntok(t): return len(tok(t, add_special_tokens=False)["input_ids"])
lens = [_ntok(t) for t in sft["train"]["text"]]
over = sum(1 for L in lens if L > MAX_LEN)
print("token len  p50=%d p95=%d p99=%d max=%d" % (np.percentile(lens,50), np.percentile(lens,95), np.percentile(lens,99), max(lens)))
print("over MAX_LEN=%d: %d (%.1f%%) -> dropped" % (MAX_LEN, over, 100*over/len(lens)))
for _s in list(sft.keys()):
    _b = len(sft[_s]); sft[_s] = sft[_s].filter(lambda ex: _ntok(ex["text"]) <= MAX_LEN)
    print("%s: kept %d/%d" % (_s, len(sft[_s]), _b))
assert len(sft["train"]) > 0 and "<think>" not in sft["train"][0]["text"]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

token len  p50=941 p95=1381 p99=4666 max=56742
over MAX_LEN=1024: 831 (39.4%) -> dropped


Filter:   0%|          | 0/2110 [00:00<?, ? examples/s]

train: kept 1279/2110


Filter:   0%|          | 0/111 [00:00<?, ? examples/s]

val: kept 71/111


In [4]:
# 4) Load base 4-bit on a SINGLE GPU + LoRA
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
# 5) SFT
from trl import SFTConfig, SFTTrainer
# Neutralize TRL's chunked-CE LM-head patch (crashes on 4-bit/device_map functools.partial forward);
# it is only a loss memory-optimization, so training is unchanged.
import trl.trainer.sft_trainer as _sftmod
try:
    import trl.trainer.dpo_trainer as _dpomod
except Exception:
    _dpomod = None
_noop = lambda *a, **k: None
for _mod in (_sftmod, _dpomod):
    if _mod is None: continue
    for _t in (_mod, getattr(_mod,"SFTTrainer",None), getattr(_mod,"DPOTrainer",None)):
        if _t is not None and hasattr(_t, "_patch_chunked_ce_lm_head"):
            setattr(_t, "_patch_chunked_ce_lm_head", _noop)
sft_cfg = SFTConfig(output_dir=f"{OUTPUT_DIR}/sft", per_device_train_batch_size=1, gradient_accumulation_steps=16,
                    num_train_epochs=SFT_EPOCHS, learning_rate=2e-4, warmup_steps=10, lr_scheduler_type="cosine",
                    fp16=True, logging_steps=20, save_strategy="steps", save_steps=50, save_total_limit=2, eval_strategy="epoch",
                    max_length=MAX_LEN, dataset_text_field="text", gradient_checkpointing=True,
                    gradient_checkpointing_kwargs={"use_reentrant": False}, optim="paged_adamw_8bit", report_to="none")
sft_trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=sft["train"], eval_dataset=sft["val"],
                         peft_config=lora, processing_class=tok)
import os
from transformers.trainer_utils import get_last_checkpoint
_ck = get_last_checkpoint(sft_cfg.output_dir) if os.path.isdir(sft_cfg.output_dir) else None
print("Resuming from", _ck) if _ck else print("No checkpoint found - starting fresh.")
sft_trainer.train(resume_from_checkpoint=_ck)
sft_trainer.save_model(f"{OUTPUT_DIR}/bpmn-sft-adapter")
tok.save_pretrained(f"{OUTPUT_DIR}/bpmn-sft-adapter")
import shutil; shutil.make_archive(f"{OUTPUT_DIR}/bpmn-sft-adapter", "zip", f"{OUTPUT_DIR}/bpmn-sft-adapter")
print("SFT adapter saved + zipped. Now Save Version (Commit) so Stage 2 can attach this output.")

Adding EOS to train dataset:   0%|          | 0/1279 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1279 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1279 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/71 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/71 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/71 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


No checkpoint found - starting fresh.


Epoch,Training Loss,Validation Loss
1,0.089659,0.095450
2,0.070969,0.080517


SFT adapter saved + zipped. Now Save Version (Commit) so Stage 2 can attach this output.


In [6]:
# 6) Quick sanity check (OPTIONAL - the adapter is already saved+zipped by cell 5).
import torch, gc
gc.collect(); torch.cuda.empty_cache()
model.config.use_cache = True     # re-enable KV cache for generation (training disabled it)
model.eval()
def chat(msg, n=256):
    txt = tok.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":msg}],
                                  tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    ids = tok(txt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False, use_cache=True)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
print(chat('What is the difference between an Error Boundary Event and an Escalation Boundary Event?'))

Error Boundary Event: An event that occurs when a process reaches a boundary where it can no longer continue (e.g., a resource is unavailable). The process is interrupted and the error is logged.

Escalation Boundary Event: An event that occurs when a process reaches a boundary where it can no longer continue (e.g., a resource is unavailable). The process is interrupted and the error is escalated to a higher-level manager or system for resolution.
